# 30 - Counterfactual impostor-risk restoration pilot

**Colab GPU, expected T4 sufficient; needs completed29 with PASSED quality gate and original23 on Drive.**
Run02 on this SAME runtime after reset; NBIS is not needed. No diffusion or new image generator architecture.

User-approved design: ALL FOUR arms start from the SAME original23 dry-only seed20261110 BEST checkpoint,
with NEW Adam optimizers, SAME192px/batch8/source/crop/dry-task/identity/order protocol,
and exactly TWO fine-tune epochs (first two frozen shuffled slices;762 updates/arm on the full data).
This is a development pilot from one predeclared historical warm-start, not seed robustness or from-scratch evidence.
The unchanged ten-condition global L1 selector includes the COMMON warm-start as epoch0.
No recognition/other-condition/test checkpoint reselection.

Arms: L1 control; L1+0.1 hardest-negative reference triplet (cosine margin0.2);
L1+0.1 mean positive paired decision-risk increase; L1+0.1 worst10% paired increase PER PROBE.
Fixed original TRAIN reference bank: one crop per identity, all SAME-DB other identities,
excluding byte-duplicate-connected groups. Identical bank in all arms; no validation/gallery/test features used for loss.
Prediction features retain input gradients through the FROZEN matcher; matcher parameters/bank/thresholds never update.
Same sigmoid temperature0.05, matched before/after negative identities and per-probe averaging.
This is an EMPIRICAL SOFT SURROGATE PENALTY, NOT a hard FMR constraint or a CVaR theorem.
Triplet is an operational reference-embedding control, not claimed reproduction of a published cGAN.

T4 expectation: unchanged restoration batch8 plus small frozen matcher, no full-image backward, no AMP/TF32.
Epoch history records peak CUDA allocated/reserved memory and time. Hardware may differ from historical runs
but library versions remain pinned, and hardware is recorded. On OOM, restart unchanged configuration on L4/A100;
do NOT silently reduce batch, add AMP, reroll seed, increase budget or substitute a checkpoint.

Safe checkpoints resume/skip completed runs; publish only when ALL FOUR finish. Existing experiments are read-only.
Run31 next: original-gallery NBIS, fixed original-input thresholds, dry three levels + identity.
Surrogate improvement alone is insufficient: it may not transfer to NBIS and may sacrifice genuine matches.
No real dry/wet labels, wet-recognition generalization, independent test, publication or robustness claim.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
import time
import types
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from google.colab import drive
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU kernel. A T4 is sufficient.')
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M1_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / 'dd1427cfad6621c577bf'
SEED = 20261110
RUN_SEEDS = {'dry_l1_seed20261110': 20261110}
RUN_NAMES = tuple(RUN_SEEDS)
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
LAMBDA_GRADIENT = 0.1
IDENTITY_FRACTION = 0.1

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact changed: {directory / name}')
print('GPU:', torch.cuda.get_device_name(0), '| One full-train dry-only L1 diagnostic; no NBIS or test evaluation.')
MATCHER_ROOT = PROJECT_ROOT / 'experiments' / 'risk_patch_matcher_v1'
RISK_ROOT = PROJECT_ROOT / 'experiments' / 'counterfactual_risk_pilot_v1'
RUN_NAMES = ('risk_l1', 'risk_triplet', 'risk_mean', 'risk_tail')
RUN_SEEDS = {name: SEED for name in RUN_NAMES}
RISK_SETTINGS = {'fine_tune_epochs': 2, 'lambda_auxiliary': 0.1, 'temperature': 0.05,
    'tail_fraction': 0.10, 'triplet_margin': 0.2, 'warm_start_seed': 20261110,
    'bank_views': 'one fixed original reference crop per train identity; all same-DB nonduplicate identities',
    'selection': 'unchanged ten-condition global L1 including shared warm-start epoch0'}


In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_config = json.loads((M1_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_completion = json.loads((M1_DIR / 'completion.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_sha, m1_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M1_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M1_DIR, m1_config['source_file_sha256'])
verify_files(M1_DIR, m1_completion['file_sha256'])
if (m1_completion['status'] != 'shared_curriculum_control_complete' or m1_completion['configuration_sha256'] != m1_sha
        or m1_config['m0_configuration_sha256'] != m0_sha):
    raise RuntimeError('Frozen completed M1 provenance mismatch.')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (m0_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m1_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen simulator/normalization provenance mismatch.')
for config in (m0_config, m1_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or np.__version__ != config['numpy_version'] or Image.__version__ != config['pillow_version']):
        raise RuntimeError('Library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M1_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M1 model/source differs from M0: {name}')
SETTINGS = dict(m1_config['settings'])
EPOCHS = len(m1_config['epoch_conditions'])
if (EPOCHS != sum(SETTINGS['phase_lengths']) or EPOCHS != m0_config['epochs']
        or SETTINGS['base_channels'] != m0_config['base_channels'] or SETTINGS['patch_size'] != m0_config['patch_size']
        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):
    raise RuntimeError('Budget/architecture/optimizer differ from the frozen baseline.')
train_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)
validation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)
for split_name, frame in (('train', train_sources), ('validation', validation_sources)):
    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')
    if frame.empty or frame['relative_path'].duplicated().any() or frame.groupby(['dataset', 'db']).ngroups != 6 or set(frame['dataset']) != {'FVC2000', 'FVC2002'}:
        raise RuntimeError(f'Invalid source-only {split_name} data.')
    for record in frame.to_dict('records'):
        relative = record['relative_path']
        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']
                or allowed.loc[relative, 'identity_key'] != record['identity_key']):
            raise RuntimeError(f'Split/source mismatch: {relative}')
        path = DATA_ROOT / relative
        if not path.is_file() or file_digest(path) != record['sha256']:
            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')
if (set(train_sources['identity_key']) & set(validation_sources['identity_key'])
        or set(train_sources['sha256']) & set(validation_sources['sha256'])
        or not validation_sources.groupby(['dataset', 'db']).size().eq(12).all()):
    raise RuntimeError('Train/validation overlap or changed validation coverage.')
training = load_module('m2_original_model_v1', M0_DIR / 'restoration_baseline.py')
simulator = load_module('m2_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
curriculum_data = load_module('m2_frozen_task_data_v1', M1_DIR / 'curriculum_data.py')
FROZEN_ENGINE_SOURCE = (M1_DIR / 'training_helpers.py.txt').read_bytes().decode('utf-8')
SOURCE_COUNT = len(train_sources)
all_conditions = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
print('Verified unchanged sources:', SOURCE_COUNT, 'train /', len(validation_sources), 'validation.')
m2_config = json.loads((M2_DIR / 'configuration.json').read_text(encoding='utf-8'))
m2_completion = json.loads((M2_DIR / 'completion.json').read_text(encoding='utf-8'))
m2_sha = file_digest(M2_DIR / 'configuration.json')
if (m2_completion['status'] != 'structure_loss_control_complete' or m2_completion['configuration_sha256'] != m2_sha
        or m2_config['m0_configuration_sha256'] != m0_sha or m2_config['m1_configuration_sha256'] != m1_sha
        or m2_config['seed'] != SEED or m2_config['settings'] != SETTINGS
        or m2_config['epoch_conditions'] != m1_config['epoch_conditions']
        or m2_config['explicit_identity_fraction'] != IDENTITY_FRACTION
        or m2_config['selection_metric'] != 'same fixed validation mean global L1; initial identity checkpoint included'):
    raise RuntimeError('Notebook13 fixed L1 recipe differs; do not run a changed specialization.')
verify_files(M2_DIR, m2_completion['file_sha256'])
verify_files(M2_DIR, m2_config['source_file_sha256'])
for key, actual in (('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),
                    ('numpy_version', np.__version__), ('pillow_version', Image.__version__),
                    ('gpu_name', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None),
                    ('gpu_compute_capability', list(torch.cuda.get_device_capability(0)) if DEVICE.type == 'cuda' else None)):
    if key not in ('gpu_name', 'gpu_compute_capability') and m2_config[key] != actual:
        raise RuntimeError(f'Dry-specialization runtime differs from original: {key}')
base_directory = M2_DIR / 'l1_control'
if file_digest(base_directory / 'completion.json') != m2_completion['run_completion_sha256']['l1_control']:
    raise RuntimeError('Original L1 completion changed.')
base_completion = json.loads((base_directory / 'completion.json').read_text(encoding='utf-8'))
verify_files(base_directory, base_completion['file_sha256'])
base_pointer = json.loads((base_directory / 'best.json').read_text(encoding='utf-8'))
if (base_completion['configuration_sha256'] != m2_sha or base_completion['run'] != 'l1_control'
        or base_completion['epochs_completed'] != EPOCHS or base_completion['result']['best_checkpoint'] != base_pointer
        or base_pointer['configuration_sha256'] != m2_sha
        or file_digest(base_directory / base_pointer['file']) != base_pointer['sha256']):
    raise RuntimeError('Original selected L1 checkpoint provenance differs.')


In [ ]:
DATA_SOURCE = '''"""Identity-task replacement around frozen source-epoch tasks."""
from operator import index
from torch.utils.data import Dataset

class PreservationPairs(Dataset):
    def __init__(self, base, identities, replacement_ids):
        self.base = base
        self.identities = identities
        self.replacement_ids = frozenset(index(task) for task in replacement_ids)
        if len(base) != len(identities) or any(task < 0 or task >= len(base) for task in self.replacement_ids):
            raise ValueError('Aligned base/identity tasks and valid replacement IDs are required.')

    def __len__(self):
        return len(self.base)

    def __getitem__(self, task_id):
        task_id = index(task_id)
        if not 0 <= task_id < len(self):
            raise IndexError(f'Invalid task ID: {task_id}')
        sample = self.identities[task_id] if task_id in self.replacement_ids else self.base[task_id]
        return {**sample, 'explicit_identity': task_id in self.replacement_ids}
'''
data_path = LOCAL_ROOT / 'm2_preservation_data.py'
data_path.parent.mkdir(parents=True, exist_ok=True)
data_path.write_text(DATA_SOURCE, encoding='utf-8')
preservation_data = load_module('m2_preservation_data_v1', data_path)
base_tasks = curriculum_data.CurriculumPairs(training, train_sources.to_dict('records'), DATA_ROOT, simulator,
    pair_config['source_dpi'], SEED, SETTINGS['patch_size'], m1_config['epoch_conditions'])
identity_tasks = curriculum_data.CurriculumPairs(training, train_sources.to_dict('records'), DATA_ROOT, simulator,
    pair_config['source_dpi'], SEED, SETTINGS['patch_size'], [[('identity', 'none')] for _ in range(EPOCHS)])
replacement_ids = []
identity_per_source_epoch = max(1, math.floor(IDENTITY_FRACTION * SOURCE_COUNT))
for epoch in range(EPOCHS):
    rng = np.random.default_rng(training.stable_seed(f'{SEED}|m2_identity|{epoch}'))
    replacement_ids.extend((epoch * SOURCE_COUNT + rng.choice(SOURCE_COUNT, identity_per_source_epoch, replace=False)).tolist())
task_dataset = preservation_data.PreservationPairs(base_tasks, identity_tasks, replacement_ids)
order = pd.read_csv(M1_DIR / 'order_shuffled_control.csv')['task_id'].to_numpy()
if not np.array_equal(np.sort(order), np.arange(len(task_dataset))):
    raise RuntimeError('Frozen shuffled task order is not a full permutation.')
orders = {name: order.copy() for name in RUN_NAMES}
validation_dataset = training.FingerprintPairs(validation_sources.to_dict('records'), DATA_ROOT, simulator,
    pair_config['source_dpi'], SEED, patch_size=SETTINGS['patch_size'], training=False)
validation_loader = DataLoader(validation_dataset, batch_size=SETTINGS['batch_size'], shuffle=False,
    num_workers=0, pin_memory=DEVICE.type == 'cuda', generator=torch.Generator().manual_seed(SEED))
print('Same tasks/order per run:', len(task_dataset), '| Updates:', EPOCHS * math.ceil(SOURCE_COUNT / SETTINGS['batch_size']))
print('Explicit identity replacements:', len(replacement_ids), '; natural identity samples may add more in late source epochs.')
expected_sources = {'restoration_baseline.py': (M0_DIR / 'restoration_baseline.py').read_bytes(),
    'train_sources.csv': (M0_DIR / 'train_sources.csv').read_bytes(),
    'validation_sources.csv': (M0_DIR / 'validation_sources.csv').read_bytes(),
    'curriculum_data.py': (M1_DIR / 'curriculum_data.py').read_bytes(),
    'preservation_data.py': DATA_SOURCE.encode(),
    'frozen_m1_engine.py.txt': FROZEN_ENGINE_SOURCE.encode(),
    'order_shared.csv': pd.DataFrame({'task_id': order}).to_csv(index=False, lineterminator='\n').encode(),
    'identity_replacement_ids.csv': pd.DataFrame({'task_id': sorted(replacement_ids)}).to_csv(index=False, lineterminator='\n').encode()}
for name, data in expected_sources.items():
    if hashlib.sha256(data).hexdigest() != m2_config['source_file_sha256'][name]:
        raise RuntimeError(f'Replication task/crop/dose or source changed: {name}')
if (len(task_dataset) != m2_config['tasks_per_run'] or identity_per_source_epoch != m2_config['explicit_identity_per_source_epoch']
        or EPOCHS * math.ceil(SOURCE_COUNT / SETTINGS['batch_size']) != m2_config['updates_per_run']):
    raise RuntimeError('Replication task budget differs from notebook13.')


In [ ]:
DRY_TASK_SOURCE = '''def specialize_conditions(epoch_conditions):
    mapped = []
    for conditions in epoch_conditions:
        if not conditions:
            raise ValueError('Every source epoch requires condition slots.')
        slots = []
        for family, severity in conditions:
            if (family == 'identity' and severity != 'none') or (family != 'identity' and
                    (family not in ('dry_like', 'wet_like', 'generic') or severity not in ('mild', 'moderate', 'severe'))):
                raise ValueError(f'Unsupported original task condition: {family}/{severity}')
            slots.append((family, severity) if family == 'identity' else ('dry_like', severity))
        mapped.append(slots)
    if not mapped:
        raise ValueError('Nonempty source epochs are required.')
    return mapped
'''
specialization = types.ModuleType('dry_task_specialization_v1')
exec(compile(DRY_TASK_SOURCE, '<frozen-dry-task-mapping>', 'exec'), specialization.__dict__)
dry_epoch_conditions = specialization.specialize_conditions(m1_config['epoch_conditions'])
original_task_dataset = task_dataset
dry_base_tasks = curriculum_data.CurriculumPairs(training, train_sources.to_dict('records'), DATA_ROOT, simulator,
    pair_config['source_dpi'], SEED, SETTINGS['patch_size'], dry_epoch_conditions)
task_dataset = preservation_data.PreservationPairs(dry_base_tasks, identity_tasks, replacement_ids)
if len(task_dataset) != len(original_task_dataset):
    raise RuntimeError('Specialization changed the number of tasks.')
ordered_position = np.empty(len(order), dtype=np.int64)
ordered_position[order] = np.arange(len(order))
audit_rows = []
source_records = train_sources.to_dict('records')
replacement_set = frozenset(replacement_ids)
for task_id in range(len(task_dataset)):
    source_epoch, source_index = divmod(task_id, SOURCE_COUNT)
    relative = source_records[source_index]['relative_path']
    original_slots = m1_config['epoch_conditions'][source_epoch]
    dry_slots = dry_epoch_conditions[source_epoch]
    if len(original_slots) != len(dry_slots):
        raise RuntimeError('Dry condition slots were deduplicated or reordered.')
    rng = np.random.default_rng(training.stable_seed(f'{SEED}|{source_epoch}|{relative}|{True}'))
    slot = int(rng.integers(len(original_slots)))
    explicit = task_id in replacement_set
    before = ('identity', 'none') if explicit else tuple(original_slots[slot])
    after = ('identity', 'none') if explicit else tuple(dry_slots[slot])
    expected = before if before[0] == 'identity' else ('dry_like', before[1])
    if after != expected:
        raise RuntimeError(f'Dry task family/severity/identity mapping differs: {task_id}')
    audit_rows.append({'task_id': task_id, 'source_epoch': source_epoch,
        'optimizer_epoch': int(ordered_position[task_id] // SOURCE_COUNT) + 1,
        'relative_path': relative, 'condition_slot': slot, 'explicit_identity': explicit,
        'original_condition': '/'.join(before), 'specialized_condition': '/'.join(after),
        'crop_seed': training.stable_seed(f'{SEED}|{source_epoch}|{relative}|crop')})
task_audit = pd.DataFrame(audit_rows)
exposure_summary = task_audit.groupby(
    ['optimizer_epoch', 'original_condition', 'specialized_condition', 'explicit_identity'], sort=True).size().reset_index(name='tasks')
if (task_audit['task_id'].duplicated().any() or len(task_audit) != len(task_dataset)
        or not task_audit['original_condition'].eq('identity/none').equals(task_audit['specialized_condition'].eq('identity/none'))
        or task_audit['explicit_identity'].sum() != len(replacement_ids)
        or not task_audit['specialized_condition'].str.split('/').str[0].isin(['dry_like', 'identity']).all()):
    raise RuntimeError('Dry specialization exposure/identity audit failed.')
specialization_protocol = {'version': 'dry_task_specialization_v1',
    'original_epoch_conditions': m1_config['epoch_conditions'], 'epoch_conditions': dry_epoch_conditions,
    'rule': 'preserve condition slots/order/multiplicity; identity unchanged; other families map to dry_like at same severity',
    'selection_unchanged': 'same fixed validation mean global L1; initial identity checkpoint included'}
specialization_data = json_bytes(specialization_protocol)
print('Specialized training task counts:')
print(task_audit.groupby(['original_condition', 'specialized_condition']).size().to_string())
print('Identity positions, severity, task order, crop seeds and total update budget preserved.')


In [ ]:
original_pointer = json.loads((SPECIALIZATION_ROOT / 'completed_experiment.json').read_text(encoding='utf-8'))
ORIGINAL_DIR = Path(original_pointer['directory'])
if (ORIGINAL_DIR.parent != SPECIALIZATION_ROOT
        or file_digest(ORIGINAL_DIR / 'configuration.json') != original_pointer['configuration_sha256']
        or file_digest(ORIGINAL_DIR / 'completion.json') != original_pointer['completion_sha256']):
    raise RuntimeError('Notebook23 completed pointer/integrity mismatch.')
original_config = json.loads((ORIGINAL_DIR / 'configuration.json').read_text(encoding='utf-8'))
original_completion = json.loads((ORIGINAL_DIR / 'completion.json').read_text(encoding='utf-8'))
if (original_completion['status'] != 'dry_specialization_complete'
        or original_completion['configuration_sha256'] != original_pointer['configuration_sha256']
        or original_config['version'] != 'm5_dry_specialization_v1'
        or original_config['initialization_seeds'] != {'dry_l1_seed20261110': 20261110}
        or original_config['m2_configuration_sha256'] != m2_sha
        or original_config['m2_completion_sha256'] != file_digest(M2_DIR / 'completion.json')
        or original_config['settings'] != SETTINGS
        or original_config['selection_metric'] != 'same fixed validation mean global L1; initial identity checkpoint included'
        or original_config['gradient_lambda'] != 0
        or original_config['losses'] != {'dry_l1_seed20261110': 'global L1'}
        or original_config['epoch_conditions'] != json.loads(json.dumps(dry_epoch_conditions))
        or original_config['original_l1_best_checkpoint'] != base_pointer):
    raise RuntimeError('Notebook23 fixed dry-only recipe differs.')
verify_files(ORIGINAL_DIR, original_completion['file_sha256'])
verify_files(ORIGINAL_DIR, original_config['source_file_sha256'])
exact_files = {**expected_sources, 'dry_task_mapping.py.txt': DRY_TASK_SOURCE.encode(),
    'dry_task_protocol.json': specialization_data,
    'task_condition_audit.csv': task_audit.to_csv(index=False, lineterminator='\n').encode(),
    'exposure_summary.csv': exposure_summary.to_csv(index=False, lineterminator='\n').encode()}
for filename, value in exact_files.items():
    if hashlib.sha256(value).hexdigest() != original_config['source_file_sha256'][filename]:
        raise RuntimeError(f'Dry initialization replication changed fixed tasks/source: {filename}')
for key, actual in (('seed', SEED), ('tasks_per_run', len(task_dataset)),
        ('updates_per_run', EPOCHS * math.ceil(SOURCE_COUNT / SETTINGS['batch_size'])),
        ('explicit_identity_fraction', IDENTITY_FRACTION),
        ('explicit_identity_per_source_epoch', identity_per_source_epoch),
        ('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),
        ('numpy_version', np.__version__), ('pillow_version', Image.__version__),
        ('gpu_name', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None),
        ('gpu_compute_capability', list(torch.cuda.get_device_capability(0)) if DEVICE.type == 'cuda' else None)):
    if key not in ('gpu_name', 'gpu_compute_capability') and original_config[key] != actual:
        raise RuntimeError(f'Original dry-only fixed recipe/runtime differs: {key}')
original_run = ORIGINAL_DIR / 'dry_l1_seed20261110'
if file_digest(original_run / 'completion.json') != original_completion['run_completion_sha256']['dry_l1_seed20261110']:
    raise RuntimeError('Original dry run completion changed.')
original_run_completion = json.loads((original_run / 'completion.json').read_text(encoding='utf-8'))
verify_files(original_run, original_run_completion['file_sha256'])
original_best = json.loads((original_run / 'best.json').read_text(encoding='utf-8'))
if (original_best != original_run_completion['result']['best_checkpoint']
        or original_best['configuration_sha256'] != original_pointer['configuration_sha256']
        or file_digest(original_run / original_best['file']) != original_best['sha256']
        or original_run_completion['epochs_completed'] != EPOCHS):
    raise RuntimeError('Original dry selected checkpoint differs.')


In [ ]:

matcher_pointer = json.loads((MATCHER_ROOT / 'completed_experiment.json').read_text(encoding='utf-8'))
MATCHER_DIR = Path(matcher_pointer['directory'])
if (MATCHER_DIR.parent != MATCHER_ROOT
        or file_digest(MATCHER_DIR / 'configuration.json') != matcher_pointer['configuration_sha256']
        or file_digest(MATCHER_DIR / 'completion.json') != matcher_pointer['completion_sha256']):
    raise RuntimeError('Notebook29 matcher pointer/integrity mismatch.')
matcher_config = json.loads((MATCHER_DIR / 'configuration.json').read_text(encoding='utf-8'))
matcher_completion = json.loads((MATCHER_DIR / 'completion.json').read_text(encoding='utf-8'))
verify_files(MATCHER_DIR, matcher_completion['file_sha256'])
gate_summary = json.loads((MATCHER_DIR / 'quality_gate.json').read_text(encoding='utf-8'))
if (matcher_completion['status'] != 'risk_matcher_fit_complete'
        or matcher_completion['configuration_sha256'] != matcher_pointer['configuration_sha256']
        or matcher_config['version'] != 'risk_patch_matcher_v1'
        or matcher_config['train_sources_sha256'] != file_digest(M0_DIR / 'train_sources.csv')
        or matcher_config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json')
        or matcher_config['simulator_sha256'] != file_digest(SIM_DIR / 'fingerprint_degradation.py')
        or matcher_config['patch_size'] != SETTINGS['patch_size']
        or matcher_completion['completed_epochs'] != matcher_config['settings']['epochs']
        or gate_summary != matcher_completion['quality_gate'] or not gate_summary['passed']):
    raise RuntimeError('MATCHER QUALITY GATE FAILED or train/protocol mismatch: STOP; do not train restoration.')
for key, actual in (('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),
        ('numpy_version', np.__version__), ('pillow_version', Image.__version__)):
    if matcher_config[key] != actual:
        raise RuntimeError(f'Frozen matcher library mismatch: {key}')
matcher_source = (MATCHER_DIR / 'matcher_source.py').read_bytes().decode()
if hashlib.sha256(matcher_source.encode()).hexdigest() != matcher_config['model_source_sha256']:
    raise RuntimeError('Matcher model source hash differs.')
matcher_module = types.ModuleType('frozen_risk_matcher')
matcher_module.__dict__.update(globals())
exec(compile(matcher_source, str(MATCHER_DIR / 'matcher_source.py'), 'exec'), matcher_module.__dict__)
matcher_checkpoint = matcher_completion['checkpoint']
if (matcher_checkpoint != json.loads((MATCHER_DIR / 'latest.json').read_text(encoding='utf-8'))
        or file_digest(MATCHER_DIR / matcher_checkpoint['file']) != matcher_checkpoint['sha256']):
    raise RuntimeError('Matcher checkpoint pointer differs.')
matcher_state = torch.load(MATCHER_DIR / matcher_checkpoint['file'], map_location='cpu', weights_only=True)
if (matcher_state['configuration_sha256'] != matcher_pointer['configuration_sha256']
        or matcher_state['completed_epochs'] != matcher_config['settings']['epochs']):
    raise RuntimeError('Matcher checkpoint payload differs.')
risk_matcher = matcher_module.PatchMatcher().to(DEVICE)
risk_matcher.load_state_dict(matcher_state['model'], strict=True)
risk_matcher.eval()
for parameter in risk_matcher.parameters():
    parameter.requires_grad_(False)
risk_thresholds = json.loads((MATCHER_DIR / 'thresholds.json').read_text(encoding='utf-8'))
if (set(risk_thresholds) != {f'{dataset}_{db}' for dataset, db in zip(train_sources.dataset, train_sources.db)}
        or not all(math.isfinite(value) and -1.001 <= value <= 1.001 for value in risk_thresholds.values())):
    raise RuntimeError('Finite fixed cosine thresholds required for all six train databases.')
del matcher_state
warm_state = torch.load(original_run / original_best['file'], map_location='cpu', weights_only=True)
if (warm_state['configuration_sha256'] != original_pointer['configuration_sha256']
        or warm_state['run'] != 'dry_l1_seed20261110' or warm_state['completed_epochs'] != original_best['completed_epochs']):
    raise RuntimeError('Shared warm-start checkpoint payload mismatch.')
warm_model_state = {key: value.detach().cpu().clone() for key, value in warm_state['model'].items()}
del warm_state
group_map = matcher_module.identity_groups(train_sources)
bank_rows = train_sources.sort_values('relative_path').drop_duplicates('identity_key')
bank_records = bank_rows.to_dict('records')
source_indices = {row['relative_path']: index for index, row in enumerate(source_records)}
with torch.inference_mode():
    features = []
    for start in range(0, len(bank_records), SETTINGS['batch_size']):
        rows = bank_records[start:start + SETTINGS['batch_size']]
        patches = torch.stack([identity_tasks[source_indices[row['relative_path']]]['target'] for row in rows]).to(DEVICE)
        features.append(risk_matcher(patches).detach())
    bank_features = torch.cat(features).detach()
# Clone outside inference_mode so frozen negatives can be saved for backward through the prediction branch.
bank_features = bank_features.clone()
if not torch.isfinite(bank_features).all():
    raise RuntimeError('Nonfinite frozen negative bank.')
source_by_path = {row['relative_path']: row for row in source_records}
print('Frozen matcher gate passed; negative identities:', len(bank_records))


In [ ]:
def gradient_loss(predictions, targets):
    if predictions.shape != targets.shape or predictions.ndim != 4 or predictions.shape[1] != 1 or min(predictions.shape[-2:]) < 3:
        raise ValueError('Sobel loss requires aligned N1HW grayscale tensors at least 3x3.')
    kernels = predictions.new_tensor([[[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
                                      [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]])[:, None] / 8
    predicted_edges = torch.nn.functional.conv2d(predictions, kernels)
    target_edges = torch.nn.functional.conv2d(targets, kernels)
    return (predicted_edges - target_edges).abs().mean()

def loss_terms(predictions, targets, name):
    if name not in RUN_NAMES:
        raise ValueError(f'Unknown objective run: {name}')
    l1 = (predictions - targets).abs().mean()
    gradient = gradient_loss(predictions, targets)
    total = l1
    return total, l1, gradient

@torch.inference_mode()
def measure_gradient(model):
    model.eval()
    totals = {}
    for batch in validation_loader:
        xs, ys = batch['input'].to(DEVICE), batch['target'].to(DEVICE)
        predictions = model(xs)
        for index, condition in enumerate(batch['condition']):
            error = float(gradient_loss(predictions[index:index + 1], ys[index:index + 1]))
            if not math.isfinite(error):
                raise RuntimeError('Non-finite validation gradient error.')
            total = totals.setdefault(condition, {'samples': 0, 'gradient_sum': 0.0})
            total['samples'] += 1
            total['gradient_sum'] += error
    if set(totals) != {f'{family}/{severity}' for family, severity in all_conditions} or any(row['samples'] != len(validation_sources) for row in totals.values()):
        raise RuntimeError('Incomplete gradient validation coverage.')
    return [{'condition': key, 'samples': value['samples'], 'gradient_mae': value['gradient_sum'] / value['samples']}
            for key, value in sorted(totals.items())]

def train_run(name):
    if name not in RUN_NAMES:
        raise ValueError(f'Unknown training run: {name}')
    initialization_seed = RUN_SEEDS[name]
    engine.SEED = initialization_seed
    directory = REPORT_DIR / name
    directory.mkdir(parents=True, exist_ok=True)
    finished = engine.load_completed_run(directory)
    if finished is not None:
        if finished['initialization_seed'] != initialization_seed:
            raise RuntimeError('Completed initialization seed differs.')
        print('Resumed completed experiment:', name, flush=True)
        return finished
    model = engine.new_model()
    model.load_state_dict(warm_model_state, strict=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=SETTINGS['learning_rate'])
    def payload(completed, history, best_mae, validation):
        return {'configuration_sha256': experiment_sha, 'run': name, 'initialization_seed': initialization_seed, 'completed_epochs': completed,
            'history': history, 'best_mae': best_mae, 'last_validation': validation,
            'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
            'torch_rng': torch.get_rng_state(), 'cuda_rng': torch.cuda.get_rng_state_all()}
    if (directory / 'latest.json').is_file():
        _, state = engine.load_checkpoint(directory, 'latest.json')
        if state['initialization_seed'] != initialization_seed:
            raise RuntimeError('Resume checkpoint initialization seed differs.')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        torch.set_rng_state(state['torch_rng'])
        torch.cuda.set_rng_state_all(state['cuda_rng'])
        completed, history, best_mae = state['completed_epochs'], state['history'], state['best_mae']
        print(f'{name}: resuming {completed}/{EPOCHS} completed epochs', flush=True)
    else:
        initial = engine.evaluate(model)
        if abs(initial['mae'] - original_best['validation_mae']) > 1e-7:
            raise RuntimeError('Shared warm-start validation differs from notebook23.')
        completed, history, best_mae = 0, [], initial['mae']
        write_verified(directory / 'initial_validation.json', json_bytes(initial))
        engine.save_checkpoint(directory, payload(0, history, best_mae, initial), True)
    if not 0 <= completed <= EPOCHS:
        raise RuntimeError('Checkpoint completed epochs exceed the configured budget.')
    for epoch in range(completed, EPOCHS):
        started = time.monotonic()
        epoch_tasks = orders[name][epoch * SOURCE_COUNT:(epoch + 1) * SOURCE_COUNT].tolist()
        loader = DataLoader(Subset(task_dataset, epoch_tasks), batch_size=SETTINGS['batch_size'], shuffle=False,
            num_workers=0, pin_memory=DEVICE.type == 'cuda', generator=torch.Generator().manual_seed(SEED + epoch))
        model.train()
        sums, risk_sums, seen, consumed, explicit_identity = np.zeros(3), np.zeros(3), 0, [], 0
        if DEVICE.type == 'cuda':
            torch.cuda.reset_peak_memory_stats()
        for step, batch in enumerate(loader, start=1):
            xs, ys = batch['input'].to(DEVICE), batch['target'].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            records = [source_by_path[path] for path in batch['relative_path']]
            total, l1, gradient, risk_values = risk_terms(model(xs), xs, ys, records, name)
            if not all(torch.isfinite(value) for value in (total, l1, gradient)):
                raise RuntimeError(f'Non-finite loss: {name}, epoch {epoch + 1}, step {step}')
            total.backward()
            if any(parameter.grad is not None and not torch.isfinite(parameter.grad).all() for parameter in model.parameters()):
                raise RuntimeError('Nonfinite restoration gradient.')
            optimizer.step()
            risk_sums += np.array([float(value.detach()) for value in risk_values.values()]) * len(xs)
            sums += np.array([float(value.detach()) for value in (total, l1, gradient)]) * len(xs)
            seen += len(xs)
            consumed.extend(batch['task_id'].tolist())
            explicit_identity += int(batch['explicit_identity'].sum())
            if step % 100 == 0:
                print(f'{name}: epoch {epoch + 1}/{EPOCHS}, step {step}/{len(loader)}, L1={sums[1] / seen:.5f}, grad={sums[2] / seen:.5f}', flush=True)
        if seen != SOURCE_COUNT or consumed != epoch_tasks:
            raise RuntimeError('Epoch did not consume the exact shared task order.')
        validation = engine.evaluate(model)
        is_best = validation['mae'] < best_mae
        best_mae = min(best_mae, validation['mae'])
        history.append({'epoch': epoch + 1, 'train_total_loss': float(sums[0] / seen), 'train_mae': float(sums[1] / seen),
            'train_gradient_loss': float(sums[2] / seen), 'validation_mae': validation['mae'],
            'validation_identity_mae': validation['identity_mae'],
            'validation_dry_mae': float(np.mean([row['mae'] for row in validation['conditions'] if row['condition'].startswith('dry_like/')])), 'explicit_identity_tasks': explicit_identity,
            'train_triplet': float(risk_sums[0] / seen), 'train_paired_mean_risk': float(risk_sums[1] / seen),
            'train_paired_tail_risk': float(risk_sums[2] / seen),
            'peak_allocated_bytes': torch.cuda.max_memory_allocated() if DEVICE.type == 'cuda' else 0,
            'peak_reserved_bytes': torch.cuda.max_memory_reserved() if DEVICE.type == 'cuda' else 0,
            'seconds': time.monotonic() - started})
        engine.save_checkpoint(directory, payload(epoch + 1, history, best_mae, validation), is_best)
        write_csv(directory / 'history.csv', pd.DataFrame(history))
        print(f'{name}: epoch {epoch + 1} saved, val={validation["mae"]:.5f}, identity={validation["identity_mae"]:.5f}, best={is_best}', flush=True)
    best_pointer, best_state = engine.load_checkpoint(directory, 'best.json')
    model.load_state_dict(best_state['model'])
    validation = engine.evaluate(model)
    if abs(validation['mae'] - best_pointer['validation_mae']) > 1e-7:
        raise RuntimeError('Selected checkpoint validation recomputation differs.')
    gradients = measure_gradient(model)
    write_csv(directory / 'history.csv', pd.DataFrame(history))
    write_verified(directory / 'best_validation.json', json_bytes(validation))
    write_csv(directory / 'best_validation_conditions.csv', pd.DataFrame(validation['conditions']))
    write_csv(directory / 'best_gradient_conditions.csv', pd.DataFrame(gradients))
    figure, axes = plt.subplots(10, 3, figsize=(9, 24))
    with torch.inference_mode():
        for row in range(10):
            sample = validation_dataset[row]
            output = model(sample['input'][None].to(DEVICE)).cpu()[0, 0].numpy()
            for column, (title, image) in enumerate(zip(('input', name, 'reference'),
                (sample['input'][0].numpy(), output, sample['target'][0].numpy()))):
                axes[row, column].imshow(np.rint(image * 255).astype(np.uint8), cmap='gray', vmin=0, vmax=255)
                axes[row, column].set_title(f'{sample["condition"]} | {title}', fontsize=7)
                axes[row, column].axis('off')
    figure.tight_layout()
    figure.savefig(directory / 'best_validation_preview.png', dpi=140)
    plt.show()
    plt.close(figure)
    result = {'run': name, 'initialization_seed': initialization_seed, 'best_epoch': best_pointer['completed_epochs'], 'epochs_completed': EPOCHS,
        'best_checkpoint': best_pointer, 'validation': validation, 'gradient_conditions': gradients,
        'recognition_improvement_verified': False, 'external_test_evaluated': False}
    latest = json.loads((directory / 'latest.json').read_text(encoding='utf-8'))
    files = {'initial_validation.json', 'history.csv', 'best_validation.json', 'best_validation_conditions.csv',
        'best_gradient_conditions.csv', 'best_validation_preview.png', 'best.json', 'latest.json', best_pointer['file'], latest['file']}
    marker = {'configuration_sha256': experiment_sha, 'run': name, 'epochs_completed': EPOCHS,
        'file_sha256': {filename: file_digest(directory / filename) for filename in sorted(files)}, 'result': result}
    write_verified(directory / 'completion.json', json_bytes(marker))
    return result

In [ ]:
RISK_SOURCE = 'def risk_terms(predictions, inputs, targets, records, name):\n    if name not in RUN_NAMES:\n        raise ValueError(f\'Unknown pilot arm: {name}\')\n    l1 = (predictions - targets).abs().mean()\n    with torch.no_grad():\n        before = risk_matcher(inputs)\n        positive = risk_matcher(targets)\n        baseline = before @ bank_features.T\n    after_features = risk_matcher(predictions)\n    after = after_features @ bank_features.T\n    valid = torch.tensor([[row[\'dataset\'] == negative[\'dataset\'] and row[\'db\'] == negative[\'db\']\n        and row[\'identity_key\'] != negative[\'identity_key\']\n        and group_map[row[\'identity_key\']] != group_map[negative[\'identity_key\']]\n        for negative in bank_records] for row in records], dtype=torch.bool, device=DEVICE)\n    if not valid.any(dim=1).all():\n        raise RuntimeError(\'Every probe needs byte-distinct different-identity negatives from the SAME DB.\')\n    cutoffs = predictions.new_tensor([risk_thresholds[f"{row[\'dataset\']}_{row[\'db\']}"] for row in records])[:, None]\n    difference = (torch.sigmoid((after - cutoffs) / RISK_SETTINGS[\'temperature\'])\n        - torch.sigmoid((baseline - cutoffs) / RISK_SETTINGS[\'temperature\'])).clamp_min(0)\n    means, tails = [], []\n    for index in range(len(records)):\n        values = difference[index][valid[index]]\n        means.append(values.mean())\n        k = max(1, math.ceil(RISK_SETTINGS[\'tail_fraction\'] * len(values)))\n        tails.append(values.topk(k, sorted=False).values.mean())\n    mean_risk, tail_risk = torch.stack(means).mean(), torch.stack(tails).mean()\n    positive_score = (after_features * positive).sum(1)\n    hardest = after.masked_fill(~valid, -torch.inf).max(1).values\n    triplet = (RISK_SETTINGS[\'triplet_margin\'] + hardest - positive_score).clamp_min(0).mean()\n    auxiliary = {\'risk_l1\': l1 * 0, \'risk_triplet\': triplet,\n        \'risk_mean\': mean_risk, \'risk_tail\': tail_risk}[name]\n    total = l1 + RISK_SETTINGS[\'lambda_auxiliary\'] * auxiliary\n    return total, l1, gradient_loss(predictions, targets), {\n        \'triplet\': triplet, \'paired_mean_risk\': mean_risk, \'paired_tail_risk\': tail_risk}\n'
exec(compile(RISK_SOURCE, '<risk-objective>', 'exec'))
EPOCHS = RISK_SETTINGS['fine_tune_epochs']


In [ ]:
helper_source = '\n\n'.join(inspect.getsource(function) for function in (gradient_loss, loss_terms, measure_gradient, train_run))
source_files = {name: (M0_DIR / name).read_bytes() for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv')}
source_files.update({'curriculum_data.py': (M1_DIR / 'curriculum_data.py').read_bytes(),
    'preservation_data.py': DATA_SOURCE.encode('utf-8'), 'dry_training_helpers.py.txt': helper_source.encode('utf-8'),
    'risk_objective.py.txt': RISK_SOURCE.encode(), 'matcher_source.py': matcher_source.encode(),
    'negative_bank.csv': bank_rows.to_csv(index=False, lineterminator='\n').encode(),
    'dry_task_mapping.py.txt': DRY_TASK_SOURCE.encode('utf-8'), 'dry_task_protocol.json': specialization_data,
    'task_condition_audit.csv': task_audit.to_csv(index=False, lineterminator='\n').encode(),
    'exposure_summary.csv': exposure_summary.to_csv(index=False, lineterminator='\n').encode(),
    'frozen_m1_engine.py.txt': FROZEN_ENGINE_SOURCE.encode('utf-8'),
    'order_shared.csv': pd.DataFrame({'task_id': order}).to_csv(index=False, lineterminator='\n').encode(),
    'identity_replacement_ids.csv': pd.DataFrame({'task_id': sorted(replacement_ids)}).to_csv(index=False, lineterminator='\n').encode()})
experiment_config = {'version': 'counterfactual_risk_pilot_v1', 'seed': SEED, 'settings': SETTINGS, 'run_names': RUN_NAMES, 'initialization_seeds': RUN_SEEDS,
    'm2_configuration_sha256': m2_sha, 'm2_completion_sha256': file_digest(M2_DIR / 'completion.json'),
    'original_l1_best_checkpoint': base_pointer,
    'warm_start_checkpoint': original_best, 'warm_start_configuration_sha256': original_pointer['configuration_sha256'],
    'matcher_configuration_sha256': matcher_pointer['configuration_sha256'],
    'matcher_completion_sha256': matcher_pointer['completion_sha256'], 'risk_settings': RISK_SETTINGS,
    'matcher_thresholds': risk_thresholds, 'negative_bank_tensor_sha256': hashlib.sha256(bank_features.cpu().numpy().astype('<f4').tobytes()).hexdigest(),
    'variation': 'objective only across four matched two-epoch fine-tunes from one predeclared common warm-start; NOT initialization robustness',
    'original_epoch_conditions': m1_config['epoch_conditions'],
    'task_mapping': specialization_protocol['rule'],
    'alternative_protocol_used_for_training_or_selection': False,
    'dry_checkpoint_reselection': False,
    'm0_configuration_sha256': m0_sha, 'm1_configuration_sha256': m1_sha,
    'simulator_configuration_sha256': file_digest(SIM_DIR / 'configuration.json'),
    'epoch_conditions': dry_epoch_conditions, 'explicit_identity_fraction': IDENTITY_FRACTION,
    'explicit_identity_per_source_epoch': identity_per_source_epoch, 'gradient_lambda': 0.0,
    'gradient_definition': 'Sobel 3x3 /8, valid convolution, mean absolute error over both directional channels',
    'losses': {'risk_l1': 'global L1', 'risk_triplet': 'L1 +0.1 hardest-negative reference triplet',
        'risk_mean': 'L1 +0.1 mean positive paired sigmoid decision-risk increase',
        'risk_tail': 'L1 +0.1 worst10% positive paired sigmoid decision-risk increase per probe'},
    'selection_metric': 'same fixed validation mean global L1; initial identity checkpoint included',
    'tasks_per_run': EPOCHS * SOURCE_COUNT, 'full_original_task_pool': len(task_dataset), 'updates_per_run': EPOCHS * math.ceil(SOURCE_COUNT / SETTINGS['batch_size']),
    'source_file_sha256': {name: hashlib.sha256(data).hexdigest() for name, data in source_files.items()},
    'torch_version': str(torch.__version__), 'torch_cuda': str(torch.version.cuda),
    'numpy_version': np.__version__, 'pillow_version': Image.__version__,
    'gpu_name': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else None,
    'gpu_compute_capability': list(torch.cuda.get_device_capability(0)) if DEVICE.type == 'cuda' else None,
    'validation_is_not_independent_test': True, 'precision': 'FP32; AMP/TF32 disabled'}
config_data = json_bytes(experiment_config)
experiment_sha = hashlib.sha256(config_data).hexdigest()
REPORT_DIR = RISK_ROOT / experiment_sha[:20]
LOCAL_EXPERIMENT = LOCAL_ROOT / 'counterfactual_risk_pilot_training' / experiment_sha[:20]
for directory in (REPORT_DIR, LOCAL_EXPERIMENT):
    directory.mkdir(parents=True, exist_ok=True)
for name, data in {**source_files, 'configuration.json': config_data}.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen structure experiment changed: {path}')
    if not path.exists():
        write_verified(path, data)
engine = types.ModuleType('frozen_m1_checkpoint_validation_engine')
engine.__dict__.update(globals())
exec(compile(FROZEN_ENGINE_SOURCE, str(M1_DIR / 'training_helpers.py.txt'), 'exec'), engine.__dict__)
print('Dry specialization frozen:', experiment_sha)
print('Drive outputs:', REPORT_DIR)

In [ ]:
results = [train_run(name) for name in RUN_NAMES]


In [ ]:

comparison = pd.DataFrame([{'method': row['run'], 'best_epoch': row['best_epoch'],
    'validation_mae': row['validation']['mae'], 'identity_mae': row['validation']['identity_mae'],
    'dry_mae': float(np.mean([condition['mae'] for condition in row['validation']['conditions']
        if condition['condition'].startswith('dry_like/')]))} for row in results])
write_csv(REPORT_DIR / 'pilot_summary.csv', comparison)
completion = dict(status='counterfactual_risk_pilot_complete', configuration_sha256=experiment_sha,
    methods=RUN_NAMES, fine_tune_epochs=EPOCHS, warm_start_checkpoint=original_best,
    matcher_quality_gate=gate_summary,
    file_sha256={name: file_digest(REPORT_DIR / name) for name in [*source_files, 'configuration.json', 'pilot_summary.csv']},
    run_completion_sha256={name: file_digest(REPORT_DIR / name / 'completion.json') for name in RUN_NAMES},
    recognition_improvement_verified=False, initialization_robustness_verified=False,
    external_test_evaluated=False)
write_verified(REPORT_DIR / 'completion.json', json_bytes(completion))
write_verified(RISK_ROOT / 'completed_experiment.json', json_bytes(dict(directory=str(REPORT_DIR),
    configuration_sha256=experiment_sha, completion_sha256=file_digest(REPORT_DIR / 'completion.json'))))
print(comparison.to_string(index=False))
print('COUNTERFACTUAL RISK PILOT TRAINING COMPLETE:', REPORT_DIR)
print('Run31 for independent NBIS evaluation. These patch/surrogate losses are not recognition evidence.')
